## Make your first Image-to-text with Gradio and Qwen2-VL model


- Image to text models output a text from a given image.

In this notebook, we will use the Qwen2-VL model which is a multimodal model that can generate text from images.


### Step 1: Install Transformers
Install the latest Transformers plus qwen-vl-utils to use Qwen2-VL.


In [ ]:
!pip install -U git+https://github.com/huggingface/transformers qwen-vl-utils


### Step 2: Import dependencies
Load the processor/model classes plus PIL, Torch, and helpers.


In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch
import requests


### Step 3: Load the Qwen2-VL model
Initialize the processor and model, then place the model on GPU for faster inference.


In [ ]:
# Follow the documentation at https://qwen2.org/vl/

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto"
)

# This code would take a while to run


While running this code, you can learn about Qwen2-VL from here
[Qwen2-VL](https://qwen2.org/vl/)


### Step 4: Run image-to-text on a sample
Fetch an image, build the chat prompt, preprocess inputs, generate text, and decode the output.


In [ ]:
url = "https://www.ilankelman.org/stopsigns/australia.jpg"  # click the link to see the image
# or this image
# url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"

image_stop = Image.open(requests.get(url, stream=True).raw)

# Display the image
image_stop.show()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": url},
            {"type": "text", "text": "What is shown in this image?"},
        ],
    },
]

# Create prompt from conversation (image + text)
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Process the image and prompt
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)
inputs = inputs.to(device)  # send inputs to CPU/GPU

generated_ids = model.generate(
    **inputs,
    max_new_tokens=100
)

# Trim the prompt tokens from the output
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True
)

print(output_text)



### Step 5: Extract the assistant answer
Trim the generated tokens to keep only the assistant response.


In [ ]:
# Filter the output text to get the answer

answer = output_text[0].strip()

print(answer)


Now, let's put everything into one function and then test our function

### Step 6: Wrap inference in a function
Create a reusable function that takes an image and a prompt (TODO: finish the body).


In [ ]:
# TODO : Try to put image-2-text in gradio platform and see the output

def generate_description(image, prompt = "What is shown in this image?", max_new_tokens=200):
    """
    Generate a description of the image using Qwen2-VL
    """
    # Guide: use the above Qwen2-VL code as the reference to write this function

    return generated_description[0]


Then serve using Gradio. `input` will be images and textbox (prompt) and output will be text (description of the text)

### Step 7: Test the function
Run a quick test with a sample image to verify the output.


In [ ]:
# Test the function that we just build
url = "https://www.ilankelman.org/stopsigns/australia.jpg" ## click on the link to see the image

image = Image.open(requests.get(url, stream=True).raw)

generate_description(
    image,
    "What is shown in this image?"
)

Note : You can use the example image from the the folder `example_images` or you can use your own image.

### Step 8: Build a Gradio demo
Create a small UI for image upload + prompt, then return the generated description using Qwen2-VL.


In [ ]:
## The output text contains the user prompt and the generated text from the model
import gradio as gr

demo = gr.Interface(
    fn=lambda img, prompt: ... ,  # put the function here
    inputs=[gr.Image(type="pil"),
            gr.Textbox(label="prompt", value="What is shown in this image?", lines=3)],
    outputs=[gr.Textbox(label="Description", lines=3)],
    title="Image Description using Qwen2-VL",
    description="Upload an image to get a detailed description using Qwen2-VL",
)

demo.launch()


### Step 9: Clean up
Close all open Gradio ports when you are done.


In [ ]:
# We can leave a lot of port open. So don't forget to close all the port using `gr.close_all()`
gr.close_all()